# Create narration data

The official TaskData test set is not used for training, validation, class balancing, threshold selection, early stopping, or architecture decisions. Only labeled development data is used. OCR/GT fields are for supervision and auditing; model input is the padded image crop only. This notebook has not been executed.

Build all accepted samples → split complete sequences → keep validation untouched → downsample non-narration inside TRAIN only. Reuse development Magi detections and OCR from `paddleOCR-Magi/magiBasePredCache.ipynb`. Missing caches cause an error; no inference runs here.

In [1]:
import hashlib
import json
import math
import random
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import torch
from PIL import Image

ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/isNarratorText')
DEVELOPMENT = ROOT.parent / 'TaskData' / 'development'
CACHE = ROOT.parent / 'paddleOCR-Magi' / 'cache_baseline' / 'development'
BBOX_PADDING = 0.25
MATCH_THRESHOLD = 95.0
MATCH_MARGIN = 5.0
TRAIN_NEGATIVE_RATIO = 2.0
VALIDATION_FRACTION = 0.20
SEED = 42


## Confident labels

Reuse the existing NFKC normalization and full fuzzy score. Require a score ≥95 and a ≥5-point margin over every other GT line. Accept direct matches or consecutive unresolved pairs. Exclude empty OCR, fallback matches, ties, and ambiguous repeated GT text. A confidently matched `UNKNOWN` speaker is non-narration under the requested binary rule. Indices are zero-based.

In [2]:
def normalize_text(text):
    text = unicodedata.normalize('NFKC', text or '').casefold()
    words = ' '.join(''.join(c if c.isalnum() else ' ' for c in text).split())
    if words:
        return words
    symbols = ''.join(c for c in text if c.isspace() or unicodedata.category(c)[0] in {'P', 'S'})
    return ' '.join(symbols.split())


def confident_match(text, gt_lines):
    normalized = normalize_text(text)
    if not normalized:
        return None
    candidates = []
    for index, gt in enumerate(gt_lines):
        target = normalize_text(gt['text'])
        if target:
            score = 100 * SequenceMatcher(None, normalized, target, autojunk=False).ratio()
            candidates.append((score, index))
    candidates.sort(reverse=True)
    if not candidates or candidates[0][0] < MATCH_THRESHOLD:
        return None
    if len(candidates) > 1 and candidates[0][0] - candidates[1][0] < MATCH_MARGIN:
        return None
    score, index = candidates[0]
    return dict(gt_index=index, gt_text=gt_lines[index]['text'],
                gt_speaker=gt_lines[index]['speaker'], match_score=score)


def match_page(ocr, gt_lines):
    matches = {}
    for index, text in enumerate(ocr):
        match = confident_match(text, gt_lines)
        if match is not None:
            matches[index] = dict(match, match_stage='direct', match_ocr_text=text)
    pairs = []
    for index in range(len(ocr) - 1):
        if index in matches or index + 1 in matches:
            continue
        if not normalize_text(ocr[index]) or not normalize_text(ocr[index + 1]):
            continue
        joined = ' '.join(ocr[index:index + 2])
        match = confident_match(joined, gt_lines)
        if match is not None:
            pairs.append((index, joined, match))
    pairs.sort(key=lambda item: item[2]['match_score'], reverse=True)
    for index, joined, match in pairs:
        if index not in matches and index + 1 not in matches:
            for text_index in (index, index + 1):
                matches[text_index] = dict(match, match_stage='adjacent_join', match_ocr_text=joined)
    return matches


def pad_bbox(bbox, image_width, image_height, ratio=BBOX_PADDING):
    x1, y1, x2, y2 = bbox
    pad_x, pad_y = ratio * (x2 - x1), ratio * (y2 - y1)
    return [max(0, x1 - pad_x), max(0, y1 - pad_y),
            min(image_width, x2 + pad_x), min(image_height, y2 + pad_y)]


def digest(value):
    text = json.dumps(value, sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


In [3]:
labels = {}
with (DEVELOPMENT / 'labels.jsonl').open(encoding='utf-8') as file:
    for line in file:
        if line.strip():
            row = json.loads(line)
            assert row['sequence_id'] not in labels
            labels[row['sequence_id']] = row

# Require cached detections for every labeled development sequence.
missing = [seq for seq in labels if not (CACHE / seq / 'magi_raw.json').is_file()]
assert not missing, f'Missing development Magi/OCR caches: {missing}'
samples = []
sequence_counts = {}
for sequence_id in sorted(labels):
    saved = json.loads((CACHE / sequence_id / 'magi_raw.json').read_text(encoding='utf-8'))
    assert saved['settings']['split'] == 'development'
    assert saved['settings']['sequence_id'] == sequence_id
    assert digest(saved['data']) == saved['data_hash'], 'Raw cache checksum mismatch.'
    page_files = sorted((DEVELOPMENT / 'images' / sequence_id).glob('*.png'))
    pages = saved['data']
    assert len(page_files) == len(pages) == len(labels[sequence_id]['pages'])
    assert [p.name for p in page_files] == [p['name'] for p in saved['settings']['pages']]
    sequence_counts[sequence_id] = dict(num_narration=0, num_not_narration=0, total_accepted=0)
    for page_idx, (page_path, page) in enumerate(zip(page_files, pages)):
        assert len(page['texts']) == len(page['ocr'])
        with Image.open(page_path) as image:
            width, height = image.size
        matches = match_page(page['ocr'], labels[sequence_id]['pages'][page_idx])
        for text_idx, match in sorted(matches.items()):
            bbox = list(map(float, page['texts'][text_idx]))
            assert len(bbox) == 4 and all(math.isfinite(x) for x in bbox)
            assert 0 <= bbox[0] < bbox[2] <= width and 0 <= bbox[1] < bbox[3] <= height
            label = int(match['gt_speaker'] == 'NARRATION')
            samples.append(dict(sequence_id=sequence_id, page_idx=page_idx,
                page_path=str(page_path), text_idx=text_idx, bbox=bbox,
                padded_bbox=pad_bbox(bbox, width, height), ocr_text=page['ocr'][text_idx],
                label=label, **match))
            counts = sequence_counts[sequence_id]
            counts['num_narration' if label else 'num_not_narration'] += 1
            counts['total_accepted'] += 1
assert samples, 'No confident development matches.'


## Split complete sequences first

Use a seeded greedy split targeting 20% of usable sequences and roughly 20% of narration and total samples. Reserve narration in both splits; never divide a sequence. If fewer than two sequences contain narration, stop with a clear message. Empty sequences are recorded but supply no samples.

In [4]:
sequence_ids = [seq for seq, counts in sequence_counts.items() if counts['total_accepted']]
assert len(sequence_ids) >= 2, 'Need at least two usable sequences.'
rng = random.Random(SEED)
rng.shuffle(sequence_ids)
positive_sequences = [seq for seq in sequence_ids if sequence_counts[seq]['num_narration']]
assert len(positive_sequences) >= 2, 'Need two narration-containing sequences; never split a sequence.'
num_validation = max(1, min(len(sequence_ids) - 1, round(len(sequence_ids) * VALIDATION_FRACTION)))
total = len(samples)
total_positive = sum(sample['label'] for sample in samples)
# Assign whole sequences while reserving narration for training.
validation_sequences = []
validation_total = validation_positive = 0
for step in range(num_validation):
    candidates = [seq for seq in sequence_ids if seq not in validation_sequences]
    if step == 0:
        candidates = [seq for seq in candidates if seq in positive_sequences]
    candidates = [seq for seq in candidates if any(
        positive not in validation_sequences + [seq] for positive in positive_sequences)]
    assert candidates, 'Cannot reserve a narration sequence for training.'
    def split_cost(seq):
        counts = sequence_counts[seq]
        total_share = (validation_total + counts['total_accepted']) / total
        positive_share = (validation_positive + counts['num_narration']) / total_positive
        return abs(total_share - VALIDATION_FRACTION) + abs(positive_share - VALIDATION_FRACTION)
    chosen = min(candidates, key=split_cost)
    validation_sequences.append(chosen)
    validation_total += sequence_counts[chosen]['total_accepted']
    validation_positive += sequence_counts[chosen]['num_narration']
train_sequences = [seq for seq in sequence_ids if seq not in validation_sequences]
splits = {
    'train': [i for i, sample in enumerate(samples) if sample['sequence_id'] in train_sequences],
    'validation': [i for i, sample in enumerate(samples) if sample['sequence_id'] in validation_sequences],
}
assert not set(train_sequences) & set(validation_sequences)
assert set(splits['train']).isdisjoint(splits['validation'])
assert set(splits['train']) | set(splits['validation']) == set(range(len(samples)))
assert all({samples[i]['label'] for i in indices} == {0, 1} for indices in splits.values())


## Downsample TRAIN only

Keep all accepted validation samples. Sample training negatives after the complete-sequence split.

In [5]:
# Sample negatives from training only after the sequence split.
positive_indices = [i for i in splits['train'] if samples[i]['label'] == 1]
negative_indices = [i for i in splits['train'] if samples[i]['label'] == 0]
rng = random.Random(SEED)
negative_subset = rng.sample(negative_indices, min(len(negative_indices),
                                                 int(TRAIN_NEGATIVE_RATIO * len(positive_indices))))
train_sample_indices = positive_indices + negative_subset
rng.shuffle(train_sample_indices)
assert set(train_sample_indices) <= set(splits['train'])
assert set(positive_indices) <= set(train_sample_indices)
assert splits['validation'] == [i for i, sample in enumerate(samples)
                                if sample['sequence_id'] in validation_sequences]
data = dict(samples=samples, splits=splits, train_sample_indices=train_sample_indices,
    sequence_splits=dict(train=train_sequences, validation=validation_sequences),
    sequence_counts=sequence_counts,
    settings=dict(bbox_padding=BBOX_PADDING, train_negative_ratio=TRAIN_NEGATIVE_RATIO,
                  seed=SEED, match_threshold=MATCH_THRESHOLD, match_margin=MATCH_MARGIN,
                  validation_fraction=VALIDATION_FRACTION, data_source='development',
                  matching='strict_direct_or_adjacent_pair', page_index_base=0))
torch.save(data, ROOT / 'narration_data.pt')
print('Total accepted development samples:', len(samples))
for title, indices in [('RAW TRAIN', splits['train']), ('ACTUAL TRAIN USED', train_sample_indices),
                       ('VALIDATION', splits['validation'])]:
    counts = Counter(samples[i]['label'] for i in indices)
    print(f'\n{title}\nNarration: {counts[1]}\nNot narration: {counts[0]}')
    print('Not narration / narration ratio:', round(counts[0] / counts[1], 3))
for split, sequences in data['sequence_splits'].items():
    print(split, 'narration-containing sequences:', sum(seq in positive_sequences for seq in sequences))
    for seq in sequences:
        print(seq, sequence_counts[seq])
print('Sequences with no accepted samples:', sum(not c['total_accepted'] for c in sequence_counts.values()))
print('Saved:', ROOT / 'narration_data.pt')


Total accepted development samples: 1624

RAW TRAIN
Narration: 60
Not narration: 1212
Not narration / narration ratio: 20.2

ACTUAL TRAIN USED
Narration: 60
Not narration: 120
Not narration / narration ratio: 2.0

VALIDATION
Narration: 19
Not narration: 333
Not narration / narration ratio: 17.526
train narration-containing sequences: 10
seq_b9b5a9ad9a64ced9 {'num_narration': 0, 'num_not_narration': 24, 'total_accepted': 24}
seq_eaad741293da2ac1 {'num_narration': 0, 'num_not_narration': 22, 'total_accepted': 22}
seq_5f9ee35846b7e635 {'num_narration': 0, 'num_not_narration': 15, 'total_accepted': 15}
seq_c2de6cb4b3c91103 {'num_narration': 0, 'num_not_narration': 20, 'total_accepted': 20}
seq_d7af6c64ebc22bbe {'num_narration': 0, 'num_not_narration': 14, 'total_accepted': 14}
seq_2032620aa4e4ac7f {'num_narration': 0, 'num_not_narration': 9, 'total_accepted': 9}
seq_b136f32c2233b97c {'num_narration': 0, 'num_not_narration': 14, 'total_accepted': 14}
seq_96839890d69d7698 {'num_narration': 0